#### Calulate % marks of each student . each subject of 100 marks. and create result with folloing conditions.




In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-q-009")
    .master("local[*]")
    .getOrCreate()
)

spark

26/03/29 12:50:56 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
students =[(1,"Steve"),(2,"David"),(3,"John"),(4,"Shree"),(5,"Helen")]

marks=[(1,"SQL",90),(1,"PySpark",100),(2,"SQL",70),(2,"PySpark",60),(3,"SQL",30),(3,"PySpark",20),(4,"SQL",50),(4,"PySpark",50),(5,"SQL",45),(5,"PySpark",45)]

student_df = spark.createDataFrame(students, ["id", "name"])
student_df.show()

marks_df = spark.createDataFrame(marks, ["id", "subject", "marks"])
marks_df.show()


+---+-----+
| id| name|
+---+-----+
|  1|Steve|
|  2|David|
|  3| John|
|  4|Shree|
|  5|Helen|
+---+-----+

+---+-------+-----+
| id|subject|marks|
+---+-------+-----+
|  1|    SQL|   90|
|  1|PySpark|  100|
|  2|    SQL|   70|
|  2|PySpark|   60|
|  3|    SQL|   30|
|  3|PySpark|   20|
|  4|    SQL|   50|
|  4|PySpark|   50|
|  5|    SQL|   45|
|  5|PySpark|   45|
+---+-------+-----+



In [4]:
input_df = student_df.join(marks_df, student_df.id == marks_df.id).drop(marks_df.id)
input_df.show()

+---+-----+-------+-----+
| id| name|subject|marks|
+---+-----+-------+-----+
|  1|Steve|    SQL|   90|
|  1|Steve|PySpark|  100|
|  2|David|    SQL|   70|
|  2|David|PySpark|   60|
|  3| John|    SQL|   30|
|  3| John|PySpark|   20|
|  4|Shree|    SQL|   50|
|  4|Shree|PySpark|   50|
|  5|Helen|    SQL|   45|
|  5|Helen|PySpark|   45|
+---+-----+-------+-----+



In [ ]:
input_df.groupBy("id",'name').sum("marks").show()

+---+-----+----------+
| id| name|sum(marks)|
+---+-----+----------+
|  1|Steve|       190|
|  2|David|       130|
|  3| John|        50|
|  4|Shree|       100|
|  5|Helen|        90|
+---+-----+----------+



In [10]:
from pyspark.sql.functions import *
df_per=input_df.groupBy('id','name').agg(
    (sum('marks')/count('*')).alias('percentage')
)

df_per.show()

+---+-----+----------+
| id| name|percentage|
+---+-----+----------+
|  1|Steve|      95.0|
|  2|David|      65.0|
|  3| John|      25.0|
|  4|Shree|      50.0|
|  5|Helen|      45.0|
+---+-----+----------+



In [ ]:
df_per.withColumn("result", 
                  when(col("percentage") >= 75, "Distinction")
                  .when((col("percentage") < 75) & (col("percentage") >= 60), "First")
                  .when((col("percentage") < 60) & (col("percentage") >= 50), "Second")
                  .when((col("percentage") < 50) & (col("percentage") >= 33), "Third")
                  .otherwise("Fail")).show()

+---+-----+----------+-----------+
| id| name|percentage|     result|
+---+-----+----------+-----------+
|  1|Steve|      95.0|Distinction|
|  2|David|      65.0|      First|
|  3| John|      25.0|       Fail|
|  4|Shree|      50.0|     Second|
|  5|Helen|      45.0|      Third|
+---+-----+----------+-----------+



In [ ]:

# Filter DataFrame for rows where 'percentage' is between 25 and 40 (inclusive)
df_per.withColumn("result", 
                    when(
                        col("percentage") >= 75, 
                        "Distinction"
                    )
                    .when(
                      col("percentage").between(60,75), 
                      "First"
                    )
                    .when(
                      col("percentage").between(51,59), 
                      "Second"
                    )
                    .when(
                      col("percentage").between(33,50), 
                      "Third"
                    )
                    .otherwise("Fail")
                ).show()